In [ ]:
code = 'SEPARATE_LEG_ITM_SL_HEDGE'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path, time_filter=False)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def SEPARATE_LEG_ITM_SL_HEDGE(bt, start_time, end_time, sell_om, hedge_om, sl):
    try:
        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
        end_dt = datetime.datetime.combine(bt.current_week_dates[-1], end_time)

        sell_ce_scrip, sell_pe_scrip, _, _, future_price, entry_dt = bt.get_strike(start_dt, end_dt, om=sell_om)
        if sell_ce_scrip is None: return None

        sell_ce_strike, sell_pe_strike = get_strike(sell_ce_scrip), get_strike(sell_pe_scrip)

        ### hedge sits hedge_om one-om units beyond the sold strike, rounded onto the strike grid
        one_om = bt.get_one_om(future_price)
        hedge_gap = round((hedge_om * one_om) / bt.gap) * bt.gap
        if hedge_gap <= 0: return None

        hedge_ce_scrip = f"{sell_ce_strike + hedge_gap}CE"
        hedge_pe_scrip = f"{sell_pe_strike - hedge_gap}PE"

        synthetic_data = bt.get_synthetic_future_data()

        ### each side stands alone - sl is measured on the sold premium only, the hedge just rides along for loss cap
        def run_side(sell_scrip, hedge_scrip, sold_strike, is_ce):

            sell_data = bt.get_single_leg_data(entry_dt, end_dt, sell_scrip)
            hedge_data = bt.get_single_leg_data(entry_dt, end_dt, hedge_scrip)
            sync_data = synthetic_data[(synthetic_data['date_time'] >= entry_dt) & (synthetic_data['date_time'] <= end_dt)]

            common_dt = np.intersect1d(sell_data['date_time'].values, hedge_data['date_time'].values)
            common_dt = np.intersect1d(common_dt, sync_data['date_time'].values)
            if len(common_dt) == 0:
                return None

            sell_data = sell_data[np.isin(sell_data['date_time'].values, common_dt)]
            hedge_data = hedge_data[np.isin(hedge_data['date_time'].values, common_dt)]
            sync_data = sync_data[np.isin(sync_data['date_time'].values, common_dt)]

            sell_dt = sell_data['date_time']
            sell_close = sell_data['close'].values
            sell_high = sell_data['high'].values
            hedge_close = hedge_data['close'].values
            sync_future = sync_data['sync_future'].values

            ### the sold leg must start OTM on the synthetic future - same yardstick the itm stop uses
            entry_sync_future = sync_future[0]
            if (entry_sync_future >= sold_strike) if is_ce else (entry_sync_future <= sold_strike):
                return None

            side_entry_time = sell_data['date_time'].iloc[0]
            sell_entry = sell_close[0]
            hedge_entry = hedge_close[0]

            ### premium stop, rounded up to a tradeable tick the way a real sell-side stoploss order would rest
            stop_price = bt.round_to_ticksize(sell_entry * (1 + (sl/100)), 'SELL', 'STOPLOSS') if sl else None

            eod_index = len(sell_close) - 1

            ### stops are scanned from the minute after entry - the entry candle's high may have printed before the fill
            ### premium stop alone - triggers on the candle high, intra minute
            sl_reason, sl_index = 'EOD', eod_index
            if stop_price is not None:
                for i in range(1, len(sell_close)):
                    if sell_high[i] >= stop_price:
                        sl_reason, sl_index = 'SL', i
                        break

            ### itm stop alone - sold strike turning ITM on the synthetic future
            itm_reason, itm_index = 'EOD', eod_index
            for i in range(1, len(sell_close)):
                if (sync_future[i] >= sold_strike) if is_ce else (sync_future[i] <= sold_strike):
                    itm_reason, itm_index = 'ITM', i
                    break

            ### slipage is charged per leg on its own entry premium
            sell_slipage = round(bt.Cal_slipage(sell_entry), 2)
            hedge_slipage = round(bt.Cal_slipage(hedge_entry), 2)

            def resolve(exit_reason, exit_index):
                fill_index = exit_index

                ### an overnight gap jumps straight past the trigger, so the open candle is not gettable either
                ### a stop of either kind hit on the day's first candle is filled at the next minute's close
                if (exit_reason != 'EOD') and (sell_dt.iloc[exit_index].time() == bt.meta_start_time):
                    if exit_index + 1 < len(sell_close):
                        fill_index = exit_index + 1
                    sell_exit = sell_close[fill_index]
                elif exit_reason == 'SL':
                    ### a resting stoploss otherwise fills at its trigger price
                    sell_exit = stop_price
                else:
                    ### itm and eod exits are market outs at the candle close
                    sell_exit = sell_close[exit_index]

                hedge_exit = hedge_close[fill_index]
                sell_pnl = round((sell_entry - sell_exit) - sell_slipage, 2)
                hedge_pnl = round((hedge_exit - hedge_entry) - hedge_slipage, 2)
                return [sell_dt.iloc[fill_index], exit_reason, sell_exit, hedge_exit, sell_pnl, hedge_pnl, round(sell_pnl + hedge_pnl, 2)]

            static = [sell_scrip, sell_entry, sell_slipage, hedge_scrip, hedge_entry, hedge_slipage, sold_strike, get_strike(hedge_scrip), side_entry_time, round(entry_sync_future, 2), stop_price if stop_price is not None else '']

            return static + resolve(sl_reason, sl_index) + resolve(itm_reason, itm_index)

        ce_side = run_side(sell_ce_scrip, hedge_ce_scrip, sell_ce_strike, True)
        if ce_side is None: return None

        pe_side = run_side(sell_pe_scrip, hedge_pe_scrip, sell_pe_strike, False)
        if pe_side is None: return None

        credit = round((ce_side[1] - ce_side[4]) + (pe_side[1] - pe_side[4]), 2)
        slipage = round(ce_side[2] + ce_side[5] + pe_side[2] + pe_side[5], 2)

        return [code, bt.index, start_time, end_time, sell_om, hedge_om, sl, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates), entry_dt, future_price, round(one_om, 2), hedge_gap] + ce_side + pe_side + [credit, slipage]

    except Exception as e:
        print(e, [bt.index, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), start_time, end_time, sell_om, hedge_om, sl])
        return

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, from_dte, to_dte, from_date, to_date, start_time, end_time, week_lists = get_meta_row_data(meta_row, pickle_path, weekly=True)

            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_SellOM/P_HedgeOM/P_SL/Start.Date/End.Date/Start.DTE/End.DTE/DayCount/EntryTime/Future/One.OM/Hedge.Gap/Sell.CE/Sell.CE.Entry/Sell.CE.Slipage/Hedge.CE/Hedge.CE.Entry/Hedge.CE.Slipage/CE.Sold.Strike/CE.Hedge.Strike/CE.Entry.Time/CE.Entry.Sync.Future/CE.Stop.Price/CE.SL.Exit.Time/CE.SL.Exit.Reason/CE.SL.Sell.Exit/CE.SL.Hedge.Exit/CE.SL.Sell.PNL/CE.SL.Hedge.PNL/CE.SL.Net.PNL/CE.ITM.Exit.Time/CE.ITM.Exit.Reason/CE.ITM.Sell.Exit/CE.ITM.Hedge.Exit/CE.ITM.Sell.PNL/CE.ITM.Hedge.PNL/CE.ITM.Net.PNL/Sell.PE/Sell.PE.Entry/Sell.PE.Slipage/Hedge.PE/Hedge.PE.Entry/Hedge.PE.Slipage/PE.Sold.Strike/PE.Hedge.Strike/PE.Entry.Time/PE.Entry.Sync.Future/PE.Stop.Price/PE.SL.Exit.Time/PE.SL.Exit.Reason/PE.SL.Sell.Exit/PE.SL.Hedge.Exit/PE.SL.Sell.PNL/PE.SL.Hedge.PNL/PE.SL.Net.PNL/PE.ITM.Exit.Time/PE.ITM.Exit.Reason/PE.ITM.Sell.Exit/PE.ITM.Hedge.Exit/PE.ITM.Sell.PNL/PE.ITM.Hedge.PNL/PE.ITM.Net.PNL/Credit/Slipage'.split('/')

            for week_dates in week_lists:
                from_date = week_dates[0]
                to_date = week_dates[-1]

                file_name = f"{index} {week_dates[0].date()} {week_dates[-1].date()} {from_dte}-{to_dte} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    wbt = WeeklyBacktest(pickle_path, index, week_dates, from_dte, to_dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SEPARATE_LEG_ITM_SL_HEDGE(wbt, row['entry_time'], row['exit_time'], row['sell_om'], row['hedge_om'], row['sl']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)

                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del wbt
                    gc.collect()

                    t2 = datetime.datetime.now()
                    print(t2-t1)

        except Exception as e:
            input(str(e))